<a href="https://colab.research.google.com/github/nermal1/Stock-Market-Prediction-437/blob/main/NaiveBayesRealPred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [38]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

## Choose tickers

In [33]:
tickers = ['AAPL', 'MSFT', '^GSPC', '^DJI']
results = {}

## feature functions

In [34]:
def weighted_moving_average(data, period):
  weights = np.arange(1, period + 1)
  return data.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

In [35]:
def featureSelection(df):
  df = df.copy()

  df['Return'] = df['Close'].pct_change()
  # simple moving average
  df['SMA_14'] = df['Close'].rolling(window=14).mean()
  df['SMA_50'] = df['Close'].rolling(window=50).mean()

  # 2. WMA (Weighted Moving Average) - 14 Day
  df['WMA_14'] = weighted_moving_average(df['Close'], 14)

  # 3. Momentum (Rate of Change over 10 days)
  df['Momentum_10'] = df['Close'] / df['Close'].shift(10) - 1

  # 4. Volatility (Rolling Standard Deviation of Returns)
  df['Volatility_14'] = df['Return'].rolling(window=14).std()

  # 5. RSI (Relative Strength Index)
  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0))
  loss = (-delta.where(delta < 0, 0))

  avg_gain = gain.rolling(window=14).mean()
  avg_loss = loss.rolling(window=14).mean()

  rs = avg_gain / avg_loss
  df['RSI_14'] = 100 - (100 / (1 + rs))

  # 6. Lags (Lagged Returns)
  lags = [1, 2, 3, 5]
  for lag in lags:
      df[f'Lag_{lag}'] = df['Return'].shift(lag)

  # 7. Create Target (1 if Up, 0 if Down)
  df['Target'] = np.where(df['Return'] > 0, 1, 0)

  # Drop NaNs created by the rolling windows (first 50 rows will be empty)
  df = df.dropna()

  return df

## Preview features

In [43]:
for ticker in tickers:
    print(f"Processing {ticker}")

    raw_df = yf.download(ticker, start="2010-01-01", end="2019-12-31", progress=False, auto_adjust=True)

    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df = raw_df.xs(ticker, axis=1, level=1)

    processed_df = featureSelection(raw_df)
    results[ticker] = processed_df

    print(f"Feature Matrix Preview ({ticker}):")
    print(results[ticker][['Close', 'RSI_14', 'WMA_14', 'Target']].tail(3))
    print("\n")

Processing AAPL
Feature Matrix Preview (AAPL):
Price           Close     RSI_14     WMA_14  Target
Date                                               
2019-12-26  69.949303  85.353119  67.690399       1
2019-12-27  69.922775  82.477188  68.094969       0
2019-12-30  70.337769  95.022066  68.511005       1


Processing MSFT
Feature Matrix Preview (MSFT):
Price            Close     RSI_14      WMA_14  Target
Date                                                 
2019-12-26  150.936203  85.355804  148.291692       1
2019-12-27  151.212021  83.286826  148.840338       1
2019-12-30  149.908813  76.375744  149.149903       0


Processing ^GSPC
Feature Matrix Preview (^GSPC):
Price             Close     RSI_14       WMA_14  Target
Date                                                   
2019-12-26  3239.909912  89.952949  3202.629020       1
2019-12-27  3240.020020  87.671045  3210.037502       1
2019-12-30  3221.290039  81.913308  3214.052367       0


Processing ^DJI
Feature Matrix Preview (^

## Naive Bayes using feature optimization

In [42]:
print("Feature Optimization Results")

feature_pool = ['RSI_14', 'SMA_14', 'SMA_50', 'WMA_14', 'Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5']

for ticker, df in results.items():
    y = df['Target']

    best_accuracy = 0
    best_combo = []
    best_metrics = {}

    # looping through all the features to see which set of 7 works best, if we looped through all the cost vs accuracy is not worth the difference.
    # Too many features can lead to overfitting which is why we limit it to 7.
    for r in range(1, 8):
        for combo in combinations(feature_pool, r):
            combo_list = list(combo)
            X = df[combo_list]

            # Split Data
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

            # Train Model
            model = GaussianNB()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # Calculate Accuracy
            acc = accuracy_score(y_test, y_pred)

            if acc > best_accuracy:
                best_accuracy = acc
                best_combo = combo_list
                best_metrics = {
                    'Precision': precision_score(y_test, y_pred, zero_division=0),
                    'Recall': recall_score(y_test, y_pred, zero_division=0),
                    'F1': f1_score(y_test, y_pred, zero_division=0)
                }

    print(f"\nResults for {ticker}:")
    print(f"  Best Accuracy: {best_accuracy:.2%}")
    print(f"  Best Features: {best_combo}")
    print(f"  Precision:     {best_metrics['Precision']:.4f}")
    print(f"  Recall:        {best_metrics['Recall']:.4f}")
    print(f"  F1 Score:      {best_metrics['F1']:.4f}")



--- Feature Optimization Results ---

Results for AAPL:
  Best Accuracy: 63.97%
  Best Features: ['SMA_50', 'WMA_14', 'Momentum_10', 'Lag_3', 'Lag_5']
  Precision:     0.6691
  Recall:        0.6790
  F1 Score:      0.6740

Results for MSFT:
  Best Accuracy: 62.75%
  Best Features: ['RSI_14', 'Momentum_10', 'Lag_5']
  Precision:     0.6420
  Recall:        0.7750
  F1 Score:      0.7023

Results for ^GSPC:
  Best Accuracy: 62.96%
  Best Features: ['Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_3', 'Lag_5']
  Precision:     0.6220
  Recall:        0.8467
  F1 Score:      0.7172

Results for ^DJI:
  Best Accuracy: 61.13%
  Best Features: ['Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5']
  Precision:     0.6176
  Recall:        0.7721
  F1 Score:      0.6863
